# MIMIC-IV Training From Extracted Features

Use this notebook when you already have `mimiciv_extracted_features_seq6.zip`.

This avoids rebuilding the full MIMIC-IV dataset from the raw ZIP. The feature ZIP should contain:

- `data/preprocessed/X_train.npy`
- `data/preprocessed/X_val.npy`
- `data/preprocessed/X_test.npy`
- `data/preprocessed/y_train.npy`
- `data/preprocessed/y_val.npy`
- `data/preprocessed/y_test.npy`
- optionally `data/real/mimiciv_longitudinal_features.csv`

Before running, set `PROJECT_DIR` to the Google Drive folder that contains this repository.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Project Folder

Change this path if your folder name is different.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/Temporal Analysis')
os.chdir(PROJECT_DIR)

print('Working directory:', Path.cwd())
print('requirements.txt exists:', (PROJECT_DIR / 'requirements.txt').exists())

## 3. Install Dependencies And Check GPU

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. Restore Extracted Feature ZIP

Put `mimiciv_extracted_features_seq6.zip` inside the project folder in Drive. If it is not there, this cell lets you upload it manually.

In [ ]:
from pathlib import Path
import zipfile

FEATURE_ZIP = PROJECT_DIR / 'mimiciv_extracted_features_seq6.zip'

if not FEATURE_ZIP.exists():
    from google.colab import files
    print('Feature ZIP not found in project folder. Upload mimiciv_extracted_features_seq6.zip now.')
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    FEATURE_ZIP = Path(uploaded_name)

print('Using feature zip:', FEATURE_ZIP)
with zipfile.ZipFile(FEATURE_ZIP, 'r') as zf:
    zf.extractall(PROJECT_DIR)

print('Extracted features into:', PROJECT_DIR)

## 5. Validate And Sanitize Arrays

This checks that the restored arrays exist and replaces any accidental `NaN` or infinite values with zero before training.

In [ ]:
from pathlib import Path
import numpy as np

p = PROJECT_DIR / 'data' / 'preprocessed'
required = [
    'X_train.npy', 'X_val.npy', 'X_test.npy',
    'y_train.npy', 'y_val.npy', 'y_test.npy',
]

missing = [name for name in required if not (p / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing required files in {p}: {missing}')

for split in ['train', 'val', 'test']:
    x_path = p / f'X_{split}.npy'
    y_path = p / f'y_{split}.npy'
    X = np.load(x_path)
    y = np.load(y_path)
    nan_count = int(np.isnan(X).sum())
    inf_count = int(np.isinf(X).sum())
    if nan_count or inf_count:
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        np.save(x_path, X)
    print(split, X.shape, 'nan=', nan_count, 'inf=', inf_count, 'pos_rate=', float(y.mean()))

print('Feature arrays are ready for training.')

## 6. Detect Training Script Layout

The cleaned repository uses `src/model_training`. Older Colab copies may use `src/models`. This cell supports both.

In [ ]:
from pathlib import Path

if (PROJECT_DIR / 'src/model_training/train_bilstm_attention.py').exists():
    TRAIN_LSTM = 'src/model_training/train_bilstm_attention.py'
    TRAIN_GRU = 'src/model_training/train_bigru.py'
    TRAIN_TRANSFORMER = 'src/model_training/train_transformer_encoder.py'
elif (PROJECT_DIR / 'src/models/train_lstm.py').exists():
    TRAIN_LSTM = 'src/models/train_lstm.py'
    TRAIN_GRU = 'src/models/train_gru.py'
    TRAIN_TRANSFORMER = 'src/models/train_transformer.py'
else:
    raise FileNotFoundError('Could not find training scripts in src/model_training or src/models.')

print('LSTM script:', TRAIN_LSTM)
print('GRU script:', TRAIN_GRU)
print('Transformer script:', TRAIN_TRANSFORMER)

## 7. Train BiLSTM Attention

In [ ]:
!python {TRAIN_LSTM} --epochs 80 --lr 0.0003 --batch_size 128 --hidden_size 128 --patience 12

## 8. Train BiGRU

In [ ]:
!python {TRAIN_GRU} --epochs 80 --lr 0.0003 --batch_size 128 --hidden_size 128 --patience 12

## 9. Train Transformer Encoder Classifier

In [ ]:
!python {TRAIN_TRANSFORMER} --epochs 80 --lr 0.0003 --batch_size 128 --hidden_size 128 --patience 12

## 10. Evaluate And Visualize

In [ ]:
from pathlib import Path

if (PROJECT_DIR / 'src/reporting/evaluate_and_visualize_all_models.py').exists():
    !python src/reporting/evaluate_and_visualize_all_models.py
elif (PROJECT_DIR / 'evaluate_trained_models.py').exists():
    !python evaluate_trained_models.py
else:
    print('No combined evaluator found. Check individual model outputs in the training logs.')